# Pose vs snippet viewer (Gradio)

Pick a **random** row from `final_dataset_v2.jsonl` that has **`pose_path`** and an on-disk **snippet video**, then show:

- **Left:** original MP4 snippet  
- **Right:** synthetic video — **black frames** with **17 colored keypoints** (COCO layout) drawn as dots  

**Requires:** `pip install gradio` (and existing deps: `opencv-python`, `numpy`, `pyyaml`).

Run all cells, then use **“Load random snippet + pose”**.

In [1]:
from __future__ import annotations

import json
import os
import random
import tempfile
from pathlib import Path

import cv2
import numpy as np
import yaml


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(14):
        if (p / "dataset_construction" / "config.yaml").is_file():
            return p
        p = p.parent
    raise FileNotFoundError("Run from repo root or a subfolder (need dataset_construction/config.yaml).")


REPO_ROOT = find_repo_root()
CONFIG_PATH = REPO_ROOT / "dataset_construction" / "config.yaml"
with open(CONFIG_PATH, encoding="utf-8") as f:
    _cfg = yaml.safe_load(f)
_fd = _cfg.get("final_dataset") or {}
_pe = _cfg.get("pose_extraction") or {}
# Default matches dataset_construction/config.yaml pose_extraction.output_dir
POSE_OUT = REPO_ROOT / str(_pe.get("output_dir", "dataset/embeddings/pose/v2"))
# Sampling rate used by `06_pose_extraction.py` (manifest `pose_actual_fps` is the *snippet* FPS, not this).
POSE_TARGET_FPS = float(_pe.get("target_fps", 5.0))
MANIFEST_PATH = REPO_ROOT / _fd["output_manifest"]

rows: list[dict] = []
with open(MANIFEST_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print(f"Manifest: {MANIFEST_PATH.relative_to(REPO_ROOT)} ({len(rows)} rows)")
print(f"Pose sampling (target_fps): {POSE_TARGET_FPS} Hz — use this for timeline, not manifest pose_actual_fps.")

Manifest: dataset_construction/manifests/final_dataset_v2.jsonl (7318 rows)
Pose sampling (target_fps): 5.0 Hz — use this for timeline, not manifest pose_actual_fps.


In [2]:
def eligible_rows() -> list[dict]:
    """Manifest rows whose snippet video exists and pose arrays exist.

    Uses ``pose_path`` / ``pose_mask_path`` from the manifest when present; otherwise looks for
    ``{snippet_id}_pose.npy`` under ``POSE_OUT`` so you can preview **before** ``06_pose_extraction.py --update-manifest``.
    """
    out: list[dict] = []
    for r in rows:
        sid = r.get("snippet_id")
        if not isinstance(sid, str):
            continue
        vp = r.get("video_path")
        if not vp or not (REPO_ROOT / vp).is_file():
            continue

        pose_abs: Path | None = None
        pp = r.get("pose_path")
        if pp and (REPO_ROOT / pp).is_file():
            pose_abs = REPO_ROOT / pp
        else:
            guess = POSE_OUT / f"{sid}_pose.npy"
            if guess.is_file():
                pose_abs = guess

        if pose_abs is None:
            continue

        row = dict(r)
        row["pose_path"] = str(pose_abs.relative_to(REPO_ROOT)).replace("\\", "/")

        mask_abs = POSE_OUT / f"{sid}_pose_mask.npy"
        mp = r.get("pose_mask_path")
        if mp and (REPO_ROOT / mp).is_file():
            row["pose_mask_path"] = str((REPO_ROOT / mp).relative_to(REPO_ROOT)).replace("\\", "/")
        elif mask_abs.is_file():
            row["pose_mask_path"] = str(mask_abs.relative_to(REPO_ROOT)).replace("\\", "/")
        else:
            row["pose_mask_path"] = None

        out.append(row)
    return out


ELIGIBLE = eligible_rows()
print(f"Pose output dir (config): {POSE_OUT.relative_to(REPO_ROOT)}")
print(f"Rows with video + pose on disk: {len(ELIGIBLE)}")
if not ELIGIBLE:
    print(
        "No matches: need snippet MP4 under video_path and "
        f"{POSE_OUT.relative_to(REPO_ROOT)}/{{snippet_id}}_pose.npy"
    )

Pose output dir (config): dataset/embeddings/pose/v2
Rows with video + pose on disk: 2616


In [3]:
# --- Visualization confidence (only draw joints/lines above these; tune as needed) ---
VIEW_MIN_JOINT_CONF = 0.35  # drop low-ViTPose joints (similar scale to YOLO gate in pose extraction)
# Set to 0 to disable: only per-joint filtering applies, frames are never cleared for being too sparse.
VIEW_MIN_JOINTS_ON_FRAME = 5  # require at least this many joints ≥ VIEW_MIN_JOINT_CONF or draw black frame

# Saturated BGR palette (easy_ViTPose postprocess swaps x/y → we draw using ch1=x, ch0=y below).
JOINT_COLORS = [
    (0, 255, 255),
    (0, 255, 128),
    (0, 255, 0),
    (255, 255, 0),
    (255, 128, 0),
    (255, 64, 0),
    (255, 0, 0),
    (255, 0, 255),
    (180, 0, 255),
    (128, 0, 255),
    (80, 0, 255),
    (0, 200, 255),
    (0, 128, 255),
    (255, 200, 255),
    (200, 255, 255),
    (255, 255, 255),
    (180, 255, 180),
]


def dot_radius_for_canvas(width: int, height: int) -> int:
    """Scale marker size with resolution (still capped for huge clips)."""
    return int(np.clip(min(width, height) // 48, 12, 28))


# Animal 17-pt layout: same as ``joints_dict()['apt36k']['skeleton']`` (easy_ViTPose vit_utils/visualization.py).
APT36K_SKELETON_EDGES: list[tuple[int, int]] = [
    (0, 1),
    (0, 2),
    (1, 2),
    (2, 3),
    (3, 4),
    (3, 5),
    (5, 6),
    (6, 7),
    (3, 8),
    (8, 9),
    (9, 10),
    (4, 11),
    (11, 12),
    (12, 13),
    (4, 14),
    (14, 15),
    (15, 16),
]
SKELETON_LINE_BGR = (210, 230, 255)


def joint_screen_positions(
    pose_row: np.ndarray,
    width: int,
    height: int,
    normalized: bool,
    conf_thr: float,
) -> list[tuple[int, int] | None]:
    """Per-joint pixel coords for drawing, or None if below ``conf_thr`` / invalid."""
    n = min(17, pose_row.shape[0])
    xc_all, yr_all = xy_image_from_pose_row(pose_row[:n])
    conf_all = pose_row[:n, 2]
    out: list[tuple[int, int] | None] = [None] * 17
    for k in range(n):
        xc, yr, c = float(xc_all[k]), float(yr_all[k]), float(conf_all[k])
        if not np.isfinite(xc) or not np.isfinite(yr) or c < conf_thr:
            continue
        if normalized:
            xi = int(np.clip(xc * height, 0, width - 1))
            yi = int(np.clip(yr * width, 0, height - 1))
        else:
            xi = int(np.clip(xc, 0, width - 1))
            yi = int(np.clip(yr, 0, height - 1))
        out[k] = (xi, yi)
    return out


def count_confident_joints(pose_row: np.ndarray, conf_thr: float) -> int:
    c = pose_row[: min(17, pose_row.shape[0]), 2]
    return int(np.sum(np.isfinite(c) & (c >= conf_thr)))


def video_size(path: Path) -> tuple[int, int]:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return max(w, 1), max(h, 1)


def video_size_from_first_frame(path: Path) -> tuple[int, int]:
    """Use decoded frame geometry (matches pose extraction better than container tags)."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        return video_size(path)
    hh, ww = frame.shape[:2]
    return max(ww, 1), max(hh, 1)


def video_frame_count(path: Path) -> int:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return 0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return max(n, 0)


def fps_from_video(path: Path, fallback: float = 5.0) -> float:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return fallback
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 0.0
    cap.release()
    return fps if fps > 1e-3 else fallback


def coords_are_normalized(pose: np.ndarray, conf_thr: float = 0.08) -> bool:
    """Heuristic on *image* x/y (after VitTPose channel order — see ``xy_image_from_pose_row``)."""
    conf = pose[:, :, 2]
    sel = np.isfinite(conf) & (conf >= conf_thr)
    xc = pose[:, :, 1]
    yr = pose[:, :, 0]
    if sel.any():
        vx = xc[sel]
        vy = yr[sel]
    else:
        m = np.isfinite(xc) & np.isfinite(yr)
        if not m.any():
            return True
        vx, vy = xc[m], yr[m]
    vmax = float(max(np.nanmax(np.abs(vx)), np.nanmax(np.abs(vy))))
    # Normalized pose uses y/W and x/H (see normalize_pose_xy + ViTPose swap); ultrawide clips can exceed 1.5.
    return vmax <= 5.0


def xy_image_from_pose_row(row: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Map stored ViTPose rows to image x (horizontal) and y (vertical).

    ``VitInference.postprocess`` concatenates ``points[:, :, ::-1]``, so heatmap (x,y) becomes
    ``[..., 0]`` = vertical coordinate, ``[..., 1]`` = horizontal coordinate in pixel space.
    """
    yr = np.asarray(row[:, 0], dtype=np.float64)
    xc = np.asarray(row[:, 1], dtype=np.float64)
    return xc, yr


def map_pose_to_native_video_frames(
    pose: np.ndarray,
    mask: np.ndarray | None,
    n_vid: int,
    fps_video: float,
    pose_sample_hz: float,
) -> tuple[np.ndarray, np.ndarray]:
    """One pose slot per native video frame so left/right clips match in duration and framing."""
    t_pose, nk, nc = pose.shape
    if mask is None:
        mask = np.ones(t_pose, dtype=bool)
    else:
        mask = np.asarray(mask, dtype=bool)
    out = np.empty((n_vid, nk, nc), dtype=np.float32)
    valid = np.empty(n_vid, dtype=bool)
    if n_vid <= 0:
        return out.reshape(0, nk, nc), valid
    if fps_video <= 1e-9:
        fps_video = 25.0
    if pose_sample_hz <= 1e-9:
        pose_sample_hz = 5.0
    for fi in range(n_vid):
        t_sec = fi / fps_video
        j = int(np.clip(np.round(t_sec * pose_sample_hz), 0, t_pose - 1))
        out[fi] = pose[j]
        valid[fi] = bool(mask[j]) if j < len(mask) else False
    return out, valid


def draw_pose_frame(
    pose_row: np.ndarray,
    mask_ok: bool,
    width: int,
    height: int,
    normalized: bool,
    conf_thr: float = VIEW_MIN_JOINT_CONF,
    radius: int | None = None,
) -> np.ndarray:
    img = np.zeros((height, width, 3), dtype=np.uint8)
    if not mask_ok:
        return img
    if count_confident_joints(pose_row, conf_thr) < VIEW_MIN_JOINTS_ON_FRAME:
        return img
    r = dot_radius_for_canvas(width, height) if radius is None else int(radius)
    outline = max(2, r // 5)
    line_th = max(3, r // 3)

    pts = joint_screen_positions(pose_row, width, height, normalized, conf_thr)

    for a, b in APT36K_SKELETON_EDGES:
        pa, pb = pts[a], pts[b]
        if pa is None or pb is None:
            continue
        cv2.line(img, pa, pb, SKELETON_LINE_BGR, line_th, lineType=cv2.LINE_AA)

    for k in range(min(17, pose_row.shape[0])):
        center = pts[k]
        if center is None:
            continue
        color = JOINT_COLORS[k % len(JOINT_COLORS)]
        cv2.circle(img, center, r + outline, (255, 255, 255), outline, lineType=cv2.LINE_AA)
        cv2.circle(img, center, r, color, -1, lineType=cv2.LINE_AA)
    return img


def render_pose_side_video(
    pose: np.ndarray,
    mask: np.ndarray | None,
    width: int,
    height: int,
    fps: float,
    out_path: Path,
    conf_thr: float = VIEW_MIN_JOINT_CONF,
) -> None:
    normalized = coords_are_normalized(pose)
    T = pose.shape[0]
    if mask is None:
        mask = np.ones(T, dtype=bool)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, float(max(fps, 1.0)), (width, height))
    if not writer.isOpened():
        raise RuntimeError("VideoWriter failed (codec); try another fourcc on your OS.")
    try:
        for t in range(T):
            ok = bool(mask[t]) if t < len(mask) else False
            frame = draw_pose_frame(pose[t], ok, width, height, normalized, conf_thr=conf_thr)
            writer.write(frame)
    finally:
        writer.release()

In [4]:
_LAST_TMP: Path | None = None


def random_snippet_and_pose(seed: int | None = None) -> tuple[str | None, str | None, str]:
    """Returns (original_video_path, pose_skeleton_video_path, caption). Paths for Gradio; ``None`` if nothing eligible."""
    global _LAST_TMP
    if not ELIGIBLE:
        return (
            None,
            None,
            "No rows with snippet video + pose .npy. "
            f"Need MP4 at video_path and {POSE_OUT.relative_to(REPO_ROOT)}/{{snippet_id}}_pose.npy "
            "(re-run the eligible-rows cell after extraction; optionally run 06_pose_extraction.py --update-manifest).",
        )
    if seed is not None:
        random.seed(seed)
    r = random.choice(ELIGIBLE)
    vid_p = REPO_ROOT / r["video_path"]
    pose_p = REPO_ROOT / r["pose_path"]
    mask_p = REPO_ROOT / r["pose_mask_path"] if r.get("pose_mask_path") else None

    pose = np.load(pose_p)
    mask = np.load(mask_p) if mask_p and mask_p.is_file() else None

    fps_v = fps_from_video(vid_p)
    n_vid = video_frame_count(vid_p)
    if n_vid <= 0:
        n_vid = int(max(round(pose.shape[0] * fps_v / POSE_TARGET_FPS), 1))

    # Same canvas as decoded frames during extraction (metadata can disagree with pixels).
    w, h = video_size_from_first_frame(vid_p)

    pose_vis, mask_vis = map_pose_to_native_video_frames(
        pose,
        mask,
        n_vid=n_vid,
        fps_video=fps_v,
        pose_sample_hz=POSE_TARGET_FPS,
    )

    if _LAST_TMP is not None and _LAST_TMP.is_file():
        try:
            _LAST_TMP.unlink()
        except OSError:
            pass
    fd, tmp_path = tempfile.mkstemp(suffix="_pose_skeleton.mp4", prefix="gradio_pose_")
    os.close(fd)
    out_tmp = Path(tmp_path)
    _LAST_TMP = out_tmp

    render_pose_side_video(pose_vis, mask_vis, w, h, fps_v, out_tmp)

    src_fps = r.get("pose_actual_fps")
    cap = (
        f"snippet_id={r.get('snippet_id')} | "
        f"pose_np {pose.shape} → video {pose_vis.shape[0]} frames @ native {fps_v:.2f} fps | "
        f"sampled keypoints @ {POSE_TARGET_FPS:g} Hz | draw conf≥{VIEW_MIN_JOINT_CONF} "
        f"(≥{VIEW_MIN_JOINTS_ON_FRAME} joints/frame) | norm_xy≈{coords_are_normalized(pose_vis)} | "
        f"(manifest pose_actual_fps={src_fps} is snippet container fps, not sample rate)"
    )
    return str(vid_p.resolve()), str(out_tmp.resolve()), cap

In [5]:
import gradio as gr

with gr.Blocks(title="Snippet vs pose") as demo:
    gr.Markdown("### Original snippet (left) · Pose dots on black (right)")
    with gr.Row():
        v_left = gr.Video(label="Original snippet", interactive=False)
        v_right = gr.Video(label="Pose skeleton (APT36K-17)", interactive=False)
    cap = gr.Textbox(label="Info", interactive=False)
    seed_in = gr.Number(label="Random seed (optional, integer)", value=None, precision=0)
    btn = gr.Button("Load random snippet + pose", variant="primary")

    def _go(seed):
        try:
            s = int(seed) if seed is not None and str(seed).strip() != "" else None
            return random_snippet_and_pose(seed=s)
        except Exception as e:
            return None, None, f"Error: {e}"

    btn.click(_go, inputs=[seed_in], outputs=[v_left, v_right, cap])

# Gradio only serves file paths inside cwd/temp unless listed here (snippet MP4s live under REPO_ROOT).
demo.launch(share=False, inline=True, allowed_paths=[str(REPO_ROOT)])

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/gradio/components/video.py:398: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/gradio/components/video.py:398: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/gradio/components/video.py:398: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/gradio/components/video.py:398: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/